# Tune + Benchmark + CIES trên Kaggle

Notebook độc lập, import thẳng vào Kaggle (Notebook -> File -> Import Notebook), không phụ thuộc
file nào khác ngoài repo GitHub. Chạy nối tiếp **3 giai đoạn**: Optuna tune (cả 5 model, 50 trial)
-> Benchmark (25 tổ hợp) -> CIES Sparkov (25 tổ hợp).

## Trước khi chạy
1. **Settings -> Internet: On** (bắt buộc, để `git clone` + `pip install`).
2. **Settings -> Accelerator: GPU** (khuyến nghị — XGBoost/CatBoost/ANN tự nhận GPU nếu có).
3. Repo phải **Public** trên GitHub để Kaggle clone ẩn danh được.
4. **Add Input**: 1 Kaggle Dataset chứa ĐỦ 4 file đã tiền xử lý:
   `train_encoded.parquet`, `test_encoded.parquet`, `train_raw.parquet`, `test_raw.parquet`
   (bản mới nhất, tạo bằng `02_preprocessing.ipynb` ở máy local — xem `KAGGLE_UPLOAD_README.md`).
5. Sửa `MODELS_SCOPE` ở cell config nếu muốn giới hạn phạm vi (mặc định: cả 5 model).

## Chạy nhiều phiên (session)
Kaggle giới hạn 9 giờ/phiên GPU. Cả 3 giai đoạn đều **tự resume** (không chạy lại phần đã xong):
- Tune: checkpoint SQLite `results/tuning_checkpoints/<model>.db`.
- Benchmark: `results/model_benchmark_results.csv` (bỏ qua tổ hợp đã có).
- CIES: `results/cies_summary_results.json` (bỏ qua tổ hợp đã có).

Hết giờ giữa chừng: **Save Version** (giữ lại `/kaggle/working`), mở phiên mới, **Add Input** bằng
chính **Notebook Output** của phiên vừa rồi (cell "Khôi phục" bên dưới tự tìm và chép lại
checkpoint + các file kết quả dở dang), chạy lại từ đầu — mọi thứ đã xong sẽ tự bị bỏ qua.

## 1. Clone code + cài dependencies

In [ ]:
import os

REPO_URL = "https://github.com/gnUrt1106/fraud-detection-CIES.git"
REPO_DIR = "/kaggle/working/fraud-detection-CIES"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull -q

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Kaggle image đã có sẵn pandas/numpy/sklearn/xgboost/torch/pyarrow. Chỉ thiếu:
# imbalanced-learn (5 kỹ thuật resample), shap (CIES), catboost, optuna (tune).
# PHẢI đặt spec trong ngoặc kép: không có thì shell hiểu ">" là chuyển hướng stdout ra file.
!pip install -q "optuna>=3.4.0" "catboost>=1.2" "pyarrow>=14.0.0" "shap>=0.43.0" "imbalanced-learn>=0.11.0"

## 2. Chuẩn bị dữ liệu

Tự động tìm cả 4 file parquet trong `/kaggle/input/**` và symlink vào `data/processed/`.

In [ ]:
import glob
import os

from src.config import PROCESSED_DATA_DIR

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

NEEDED = ["train_encoded.parquet", "test_encoded.parquet", "train_raw.parquet", "test_raw.parquet"]
missing = []
for name in NEEDED:
    candidates = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not candidates:
        missing.append(name)
        continue
    dst = PROCESSED_DATA_DIR / name
    if not dst.exists():
        os.symlink(candidates[0], dst)
    print(f"Đã liên kết: {candidates[0]} -> {dst}")

if missing:
    raise FileNotFoundError(
        f"KHÔNG tìm thấy trong /kaggle/input: {missing}\n"
        "-> Kiểm tra lại đã Add Input đúng dataset chứa đủ 4 file train/test_encoded + "
        "train/test_raw (bản mới nhất từ 02_preprocessing) chưa."
    )

## 3. Config — phạm vi model, số trial, ngân sách thời gian

In [ ]:
import time

from src.config import MODEL_NAMES, IMBALANCE_TECHNIQUES

# Mặc định tune + benchmark + CIES cho CẢ 5 model (kể cả ANN). Sửa danh sách này nếu muốn
# giới hạn phạm vi 1 phiên chạy.
MODELS_SCOPE = MODEL_NAMES

N_TRIALS = 50       # vừa quota GPU Kaggle (30h/tuần) cho cả 5 model + benchmark + CIES
N_SPLITS = 5
SUBSAMPLE_N = 100_000  # cỡ mẫu bootstrap cho CIES (giống notebook 04 local)

# Ngân sách CHUNG cho cả phiên (Kaggle GPU giới hạn 9h/phiên; quá giới hạn thì phiên bị giết và
# có thể mất output). Mọi giai đoạn chỉ BẮT ĐẦU việc mới khi còn đủ giờ; việc đang chạy không bị cắt.
SESSION_START = time.time()
TOTAL_BUDGET = 7.5 * 3600

# Giờ tối thiểu phải còn lại để bắt đầu 1 tổ hợp benchmark/CIES. Ước lượng thô từ lần chạy local
# (10 nhân): SMOTE-ENN chậm nhất (~1h20 benchmark CatBoost trên full 1.48M dòng); Kaggle chỉ có ~4
# nhân CPU cho bước resample nên để dư. Chỉnh nếu thấy phiên đầu chạy nhanh/chậm hơn nhiều.
MIN_HOURS_TO_START = {"smote_enn": 3.0}
DEFAULT_MIN_HOURS_TO_START = 1.0


def time_left():
    return TOTAL_BUDGET - (time.time() - SESSION_START)


def can_start(technique):
    need = MIN_HOURS_TO_START.get(technique, DEFAULT_MIN_HOURS_TO_START) * 3600
    return time_left() >= need


print("Models:", MODELS_SCOPE)
print("Techniques:", IMBALANCE_TECHNIQUES)

### Reset kết quả của repo + khôi phục tiến độ phiên trước

`git clone` mang theo `results/best_params.json`, `model_benchmark_results.csv`,
`cies_summary_results.json` đã commit trong repo (tính bằng tham số CŨ). Nếu giữ lại, benchmark/CIES
sẽ tưởng mọi tổ hợp "đã có" và bỏ qua, còn ANN sẽ dùng tham số 30 trial cũ. Cell dưới xoá 3 file đó
**một lần mỗi phiên** (chỉ trong bản clone trên Kaggle, không đụng repo), rồi chép lại tiến độ của
phiên trước nếu bạn Add Input bằng Notebook Output. Chạy lại cell trong cùng phiên không xoá gì thêm.

In [ ]:
import glob
import shutil

from src.config import RESULTS_DIR

CKPT_DIR = RESULTS_DIR / "tuning_checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULT_FILES = ["best_params.json", "model_benchmark_results.csv", "cies_summary_results.json"]
marker = RESULTS_DIR / ".kaggle_session_initialized"

if marker.exists():
    print("Đã khởi tạo trong phiên này — giữ nguyên tiến độ hiện có.")
else:
    for name in RESULT_FILES:
        (RESULTS_DIR / name).unlink(missing_ok=True)

    restored = []
    for f in glob.glob("/kaggle/input/**/tuning_checkpoints/*.db", recursive=True):
        dst = CKPT_DIR / os.path.basename(f)
        shutil.copy(f, dst)
        restored.append(str(dst))
    for name in RESULT_FILES:
        found = sorted(glob.glob(f"/kaggle/input/**/results/{name}", recursive=True))
        if found:
            shutil.copy(found[0], RESULTS_DIR / name)
            restored.append(str(RESULTS_DIR / name))

    marker.touch()
    if restored:
        print("Đã khôi phục từ phiên trước:", *restored, sep="\n  ")
    else:
        print("Không có tiến độ phiên trước — bắt đầu từ đầu (đã bỏ kết quả cũ của repo).")

## 4. Giai đoạn A — Optuna tune (50 trial/model)

Dùng `tune_all_models` gốc (qua `run_isolated`/`multiprocessing spawn`) — trên Linux của Kaggle
cách này chạy ổn định. Kết quả tự merge vào `results/best_params.json`, chỉ ghi khi ĐỦ N_TRIALS.

In [ ]:
import pandas as pd
from src.config import PROCESSED_DATA_DIR, TARGET_COL

train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train_encoded.parquet")
X = train_df.drop(columns=[TARGET_COL]).values
y = train_df[TARGET_COL].values
print(f"Train shape: {X.shape}, fraud rate: {y.mean():.4%}")

In [ ]:
from src.models.tune import tune_all_models

tune_results = tune_all_models(
    X, y,
    models=MODELS_SCOPE,
    n_trials=N_TRIALS,
    n_splits=N_SPLITS,
    checkpoint_dir=CKPT_DIR,
    session_budget=max(time_left(), 0),
)
for model_name, res in tune_results.items():
    if "best_pr_auc" in res:
        print(f"{model_name}: best PR-AUC = {res['best_pr_auc']:.4f}  params = {res['best_params']}")
    else:
        print(f"{model_name}: chưa xong (xem log phía trên) — chạy lại phiên sau để tiếp tục.")
print(f"Còn {time_left() / 3600:.2f} giờ ngân sách.")

## 5. Giai đoạn B — Benchmark (model × kỹ thuật, `results/model_benchmark_results.csv`)

Chỉ chạy cho model đã có `best_params.json` ĐỦ N_TRIALS (model chưa tune xong ở Giai đoạn A sẽ
tự dùng tham số mặc định nếu chạy — để tránh benchmark bằng tham số chưa tối ưu, cell dưới lọc
`MODELS_SCOPE` theo model đã có trong `best_params.json`, trừ khi bạn ép qua `FORCE_UNTUNED`.

In [ ]:
import json

FORCE_UNTUNED = False  # True nếu muốn benchmark cả model chưa tune xong (dùng tham số mặc định)

best_params_path = RESULTS_DIR / "best_params.json"
tuned_models = set()
if best_params_path.exists():
    bp = json.load(open(best_params_path, encoding="utf-8"))
    tuned_models = {m for m, v in bp.items() if "best_params" in v}

BENCH_MODELS = MODELS_SCOPE if FORCE_UNTUNED else [m for m in MODELS_SCOPE if m in tuned_models]
skipped = [m for m in MODELS_SCOPE if m not in BENCH_MODELS]
if skipped:
    print(f"Bỏ qua (chưa tune xong): {skipped} — đặt FORCE_UNTUNED=True nếu muốn benchmark bằng tham số mặc định.")
print("Benchmark cho:", BENCH_MODELS)

In [ ]:
from src.utils.isolation import run_isolated
from src.models.train import train_and_evaluate_combo
from src.imbalance.resamplers import onehot_groups_from_columns

test_df = pd.read_parquet(PROCESSED_DATA_DIR / "test_encoded.parquet")
onehot_groups = onehot_groups_from_columns([c for c in train_df.columns if c != TARGET_COL])

X_train_raw = train_df.drop(columns=[TARGET_COL]).values
y_train_raw = train_df[TARGET_COL].values
X_test = test_df.drop(columns=[TARGET_COL]).values
y_test = test_df[TARGET_COL].values
print(f"Train: {X_train_raw.shape}, Test: {X_test.shape}, test fraud rate: {y_test.mean():.4%}")

In [ ]:
RESULT_CSV = RESULTS_DIR / "model_benchmark_results.csv"

benchmark_results = []
if RESULT_CSV.exists():
    benchmark_results = pd.read_csv(RESULT_CSV).to_dict("records")
    print(f"Đã có {len(benchmark_results)} tổ hợp từ trước — chỉ chạy tổ hợp còn thiếu.")
done = {(r["model"], r["imbalance_technique"]) for r in benchmark_results}

out_of_time = False
for model_name in BENCH_MODELS:
    for technique in IMBALANCE_TECHNIQUES:
        combo_name = f"{model_name} + {technique}"
        if (model_name, technique) in done:
            print(f"Bỏ qua (đã có): {combo_name}")
            continue
        if not can_start(technique):
            print(f"⏸ Còn {time_left() / 3600:.2f}h — không đủ để bắt đầu {combo_name}. Save Version rồi chạy phiên sau.")
            out_of_time = True
            break
        print(f"--> Training: {combo_name}...")
        try:
            row = run_isolated(
                train_and_evaluate_combo,
                model_name, technique,
                X_train_raw, y_train_raw, X_test, y_test,
                onehot_groups=onehot_groups,
                timeout=None,
            )
            benchmark_results.append(row)
            print(f"    PR-AUC: {row['pr_auc']:.4f} | F1: {row['f1']:.4f} | F2: {row['f2']:.4f}")
            pd.DataFrame(benchmark_results).to_csv(RESULT_CSV, index=False)
        except Exception as e:
            print(f"    Lỗi khi chạy {combo_name}: {e}")
    if out_of_time:
        break

print("Benchmark: dừng vì hết giờ." if out_of_time else "Benchmark hoàn tất (hoặc đã bỏ qua hết tổ hợp đã có).")

## 6. Giai đoạn C — CIES Sparkov (model × kỹ thuật, `results/cies_summary_results.json`)

Bootstrap resample trên mẫu con 100.000 dòng (như notebook 04 local), 20 run/tổ hợp. Ghi qua
`merge_cies_result` — an toàn nếu bạn chạy song song nhiều phiên/tiến trình khác nhau.

In [ ]:
from sklearn.model_selection import train_test_split
from src.config import SEED, N_RUNS
from src.explainability.cies import run_cies_experiment_isolated, merge_cies_result

train_raw = pd.read_parquet(PROCESSED_DATA_DIR / "train_raw.parquet")
test_raw = pd.read_parquet(PROCESSED_DATA_DIR / "test_raw.parquet")

eval_fraud = test_raw[test_raw[TARGET_COL] == 1].sample(n=min(50, int(test_raw[TARGET_COL].sum())), random_state=SEED)
eval_legit = test_raw[test_raw[TARGET_COL] == 0].sample(n=min(200, int((test_raw[TARGET_COL] == 0).sum())), random_state=SEED)
df_test_fixed_eval = pd.concat([eval_fraud, eval_legit]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"Fixed Eval Set: {len(df_test_fixed_eval)} mẫu (Fraud: {df_test_fixed_eval[TARGET_COL].sum()})")

def stratified_subsample(df, n):
    if n >= len(df):
        return df
    sub, _ = train_test_split(df, train_size=n, stratify=df[TARGET_COL], random_state=SEED)
    return sub.reset_index(drop=True)

train_sub = stratified_subsample(train_raw, SUBSAMPLE_N)
print(f"Train cho bootstrap: {len(train_sub):,} dòng (fraud: {int(train_sub[TARGET_COL].sum())})")

In [ ]:
CIES_OUT = "cies_summary_results.json"
cies_path = RESULTS_DIR / CIES_OUT

done_combos = set()
if cies_path.exists():
    done_combos = {
        (r.get("model_name"), r.get("imbalance_technique"))
        for r in json.load(open(cies_path, encoding="utf-8"))
        if "cies_metrics" in r
    }
    print(f"Đã có {len(done_combos)} tổ hợp CIES hợp lệ từ trước — chỉ chạy tổ hợp còn thiếu.")

# Giống Giai đoạn B: chỉ chạy CIES cho model đã tune xong, trừ khi FORCE_UNTUNED.
CIES_MODELS = MODELS_SCOPE if FORCE_UNTUNED else [m for m in MODELS_SCOPE if m in tuned_models]

out_of_time = False
for model_name in CIES_MODELS:
    for technique in IMBALANCE_TECHNIQUES:
        if (model_name, technique) in done_combos:
            print(f"Bỏ qua (đã có): {model_name} x {technique}")
            continue
        if not can_start(technique):
            print(f"⏸ Còn {time_left() / 3600:.2f}h — không đủ để bắt đầu {model_name} x {technique}. Save Version rồi chạy phiên sau.")
            out_of_time = True
            break
        print(f"\n=== CIES: {model_name} x {technique} ===")
        try:
            res = run_cies_experiment_isolated(
                model_name=model_name,
                imbalance_technique=technique,
                df_train=train_sub,
                df_test_fixed_eval=df_test_fixed_eval,
                target_col=TARGET_COL,
                n_runs=N_RUNS,
                feature_level=True,
                timeout=6 * 3600,
            )
            m = res["cies_metrics"]
            print(f"--> CIES={m['cies_score']:.4f} (Spearman={m['mean_spearman']:.4f}, runs={m['n_runs']})")
        except (TimeoutError, RuntimeError) as e:
            print(f"  Thất bại: {type(e).__name__}: {e}")
            res = {"model_name": model_name, "imbalance_technique": technique, "error": f"{type(e).__name__}: {e}"}
        merge_cies_result(res, RESULTS_DIR, filename=CIES_OUT)
    if out_of_time:
        break

print("CIES: dừng vì hết giờ." if out_of_time else "CIES hoàn tất (hoặc đã bỏ qua hết tổ hợp đã có).")

## 7. Kết quả

Mọi file nằm trong `/kaggle/working/fraud-detection-CIES/results/` — **Save Version** để Kaggle
giữ lại `/kaggle/working`, tải về qua tab Output:
- `best_params.json` — tham số đã tune (50 trial/model).
- `model_benchmark_results.csv` — PR-AUC/F1/F2 25 tổ hợp.
- `cies_summary_results.json` — CIES 25 tổ hợp (ghi đè `results/cies_summary_results.json` ở repo local khi tải về).
- `tuning_checkpoints/*.db` — chỉ cần giữ nếu còn phiên chưa xong ở Giai đoạn A.

Nếu hết giờ phiên giữa chừng ở bất kỳ giai đoạn nào: Save Version, mở phiên mới, Add Input bằng
Notebook Output của phiên này, chạy lại từ đầu notebook — mọi phần đã xong tự động bị bỏ qua.